# 📋 AR Follow-up Activity Simulation

> **Notebook:** `user_comments_generator.ipynb`  
> **Stage:** 2 of 2 — Collection Activity Layer  
> **Input:** `data/raw/AR_Invoices_950K.csv`  
> **Output:** `data/output/User_Comments.xlsx`

This notebook simulates realistic **AR collection team activity** by:
1. Loading the 950K invoice dataset generated by `data_generator.ipynb`
2. Filtering for **Open** and **Partial** invoices that require follow-up
3. Sampling 5,000 invoices to simulate a realistic team workload
4. Generating human-like follow-up notes, promise-to-pay dates, and collector assignments
5. Exporting the activity log to Excel for ERP upload simulation

---
## ⚙️ Setup & Path Configuration

We use `pathlib` for OS-agnostic paths — no hardcoded backslashes that cause `SyntaxWarning` on Windows.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import sys

# ── Resolve project root (works both in Jupyter and as a script) ──────────────
NOTEBOOK_DIR = Path.cwd()  # notebooks/ when run from Jupyter
PROJECT_ROOT = NOTEBOOK_DIR.parent

# ── Key directories ───────────────────────────────────────────────────────────
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR   = PROJECT_ROOT / 'data' / 'output'

# Auto-create output folder if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set seed for reproducibility
np.random.seed(42)

print('✅ Environment ready')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Raw data     : {RAW_DATA_DIR}')
print(f'   Output dir   : {OUTPUT_DIR}')

---
## 📥 Step 1 — Load Open Invoices

We load `AR_Invoices_950K.csv` and filter for invoices with a status of **`Open`** or **`Partial`**.  
These represent accounts that the collection team must actively follow up on.

> **Expected distribution:** ~40% Open · ~20% Partial → ~570,000 invoices eligible for follow-up.

In [ ]:
# ── File path via pathlib — no backslash SyntaxWarnings ──────────────────────
file_path = RAW_DATA_DIR / 'AR_Invoices_950K.csv'

print(f'Looking for file at:\n  {file_path}\n')

if file_path.exists():
    invoices_df = pd.read_csv(file_path, parse_dates=['PostingDate'])

    # Filter for invoices requiring collection follow-up
    open_mask    = invoices_df['Status'].isin(['Open', 'Partial'])
    open_invoices = invoices_df[open_mask]['InvoiceID'].values

    print('📊 Dataset Summary')
    print('─' * 40)
    print(f'  Total invoices         : {len(invoices_df):>10,}')
    print(f'  Open invoices (40%)    : {(invoices_df["Status"]=="Open").sum():>10,}')
    print(f'  Partial invoices (20%) : {(invoices_df["Status"]=="Partial").sum():>10,}')
    print(f'  Cleared invoices (40%) : {(invoices_df["Status"]=="Cleared").sum():>10,}')
    print('─' * 40)
    print(f'  ➡ Eligible for follow-up: {len(open_invoices):>9,}')
else:
    raise FileNotFoundError(
        f"\n❌ File not found: {file_path}"
        f"\n\n📂 Please generate the data first by running:"
        f"\n   notebooks/data_generator.ipynb"
        f"\n\n   The file will be saved to: {RAW_DATA_DIR}"
    )

---
## 🎲 Step 2 — Sample for Activity Simulation

Instead of generating activity for every eligible invoice, we sample **5,000** records —  
simulating a realistic daily/weekly workload for an AR collection team.

In [ ]:
SAMPLE_SIZE = 5_000

if len(open_invoices) >= SAMPLE_SIZE:
    sampled_invoices = np.random.choice(open_invoices, SAMPLE_SIZE, replace=False)
    print(f'✅ Sampled {SAMPLE_SIZE:,} invoices from {len(open_invoices):,} eligible records.')
else:
    sampled_invoices = open_invoices
    print(
        f'⚠️  Warning: Only {len(open_invoices):,} eligible invoices found. '
        f'Using all of them (requested: {SAMPLE_SIZE:,}).'
    )

---
## 💬 Step 3 — Generate Synthetic Follow-up Comments

We simulate the human-generated data that a collector would typically enter into an ERP system:

| Field | Description |
|-------|-------------|
| `FollowUpNote` | One of 4 realistic follow-up templates |
| `PromisedDate` | Customer's promise-to-pay (1–14 days from today) |
| `CollectorName` | Team member who made the contact |
| `ContactDate` | Today's date (when the contact was made) |

In [ ]:
# ── Follow-up note templates ──────────────────────────────────────────────────
follow_up_templates = [
    'Customer promised to pay next Thursday',
    'Waiting for manager approval on payment',
    'Dispute over quantity — reviewing with sales team',
    'Payment initiated, waiting for bank clearance',
    'Customer requested invoice copy — resent via email',
    'No response — left voicemail, follow up tomorrow',
    'Partial payment received — remainder due next week',
    'Escalated to senior collector for overdue account',
]

# ── Collector team ────────────────────────────────────────────────────────────
collectors = ['Ahmed', 'Sara', 'Omar', 'Nour', 'Yasmin']

# ── Build the activity DataFrame ──────────────────────────────────────────────
today = datetime.today().date()

comments_df = pd.DataFrame({
    'InvoiceID'    : sampled_invoices,
    'ContactDate'  : today,
    'FollowUpNote' : np.random.choice(follow_up_templates, len(sampled_invoices)),
    'PromisedDate' : [
        today + timedelta(days=int(d))
        for d in np.random.randint(1, 15, len(sampled_invoices))
    ],
    'CollectorName': np.random.choice(collectors, len(sampled_invoices)),
})

print(f'✅ Generated {len(comments_df):,} follow-up records')
print()
print('📋 Preview (first 5 rows):')
display(comments_df.head())

print('\n📊 Collector Workload Distribution:')
display(
    comments_df['CollectorName']
    .value_counts()
    .rename_axis('Collector')
    .reset_index(name='Invoices Handled')
)

print('\n📊 Follow-up Note Distribution:')
display(
    comments_df['FollowUpNote']
    .value_counts()
    .rename_axis('Follow-up Note')
    .reset_index(name='Count')
)

---
## 💾 Step 4 — Export to Excel

The final output is saved to `data/output/User_Comments.xlsx`.  
This file simulates the **manual input file** typically used to bulk-upload follow-up notes back into an ERP or AR system (e.g., SAP FI/AR).

In [ ]:
output_file = OUTPUT_DIR / 'User_Comments.xlsx'

comments_df.to_excel(output_file, index=False)

print(f'✅ Export complete!')
print(f'   Records  : {len(comments_df):,}')
print(f'   Saved to : {output_file}')